In [176]:
import numpy as np

In [177]:
wx = 0.5
wh = 0.8
wy = 1.0

bh = 0.0
by = 0.0

x = [1,2,3]

h_prev = 0

hs = [0]
ys = []

for xt in x:

    h = np.tanh(wx*xt + wh*h_prev +bh)
    y = wy*h + by

    hs.append(h)
    ys.append(y)

    h_prev = h

print("hidden:", hs)
print("output:", ys)

hidden: [0, np.float64(0.46211715726000974), np.float64(0.8786223746838898), np.float64(0.9758816208890569)]
output: [np.float64(0.46211715726000974), np.float64(0.8786223746838898), np.float64(0.9758816208890569)]


In [178]:
# x1 -> h1 -> y1
# x2 -> h2 -> y2
# x3 -> h3 -> y3

In [179]:
target = [0,0,1]

loss = 0

for y, t in zip(ys, target):
    loss += 0.5*(y-t)**2

print(loss)

0.4930556202700847


In [180]:
n = 3

dh_next = 0
dWy = 0
dby = 0
dh = 0
dWx= 0
dWh= 0
dbh = 0


for t in range(n - 1, -1, -1):

    dy = ys[t] - target[t] 

    dWy += dy * hs[t] 
    dby += dy

    dh = wy * dy
    dh += dh_next

    da = dh * (1 - hs[t]**2)

    dWx += da * x[t]
    dWh += da * hs[t-1]
    dbh += da

    dh_next = da * wh

In [181]:
class RNNNeuron:

    def __init__(self):

        self.wx = np.random.randn() * 0.1
        self.wh = np.random.randn() * 0.1
        self.wy = np.random.randn() * 0.1

        self.b = 0.0

    def forward(self, xs):

        self.xs = xs
        self.zs = []
        self.hs = [0.0]  # h0
        self.ys = []

        for xt in xs:

            h_prev = self.hs[-1]

            z = self.wx * xt + self.wh * h_prev + self.b
            h = np.tanh(z)
            y = self.wy * h

            self.zs.append(z)
            self.hs.append(h)
            self.ys.append(y)

        return self.ys

    def compute_loss(self, targets):

        loss = 0

        for y, t in zip(self.ys, targets):

            loss += 0.5 * (y - t) ** 2

        return loss

    def backward(self, targets):

        self.dwx = 0
        self.dwh = 0
        self.dwy = 0
        self.db = 0

        dh_next = 0

        T = len(self.xs)

        for t in reversed(range(T)):

            xt = self.xs[t]

            ht = self.hs[t + 1]
            h_prev = self.hs[t]

            yt = self.ys[t]

            target = targets[t]

            # dL/dy
            dy = yt - target

            # wy gradient
            self.dwy += dy * ht

            # total gradient arriving at h
            dh = dy * self.wy + dh_next

            # tanh derivative
            dz = dh * (1 - ht ** 2)

            # parameter gradients
            self.dwx += dz * xt
            self.dwh += dz * h_prev
            self.db += dz

            # send gradient backward in time
            dh_next = dz * self.wh

    def step(self, lr):

        self.wx -= lr * self.dwx
        self.wh -= lr * self.dwh
        self.wy -= lr * self.dwy
        self.b -= lr * self.db


In [182]:
rnn = RNNNeuron()

xs = [1, 2, 3]
targets = [0, 0, 1]

for epoch in range(500):

    preds = rnn.forward(xs)

    loss = rnn.compute_loss(targets)

    rnn.backward(targets)

    rnn.step(0.01)

    if epoch % 50 == 0:

        print(f"epoch={epoch:3d}",f"loss={loss:.6f}")


print("\nPredictions:")
preds = rnn.forward(xs)

for p in preds:
    print(round(float(p), 4))

print("\nWeights:")
print("wx =", rnn.wx)
print("wh =", rnn.wh)
print("wy =", rnn.wy)
print("b  =", rnn.b)

epoch=  0 loss=0.506401
epoch= 50 loss=0.363238
epoch=100 loss=0.264442
epoch=150 loss=0.237137
epoch=200 loss=0.219884
epoch=250 loss=0.202178
epoch=300 loss=0.183717
epoch=350 loss=0.165068
epoch=400 loss=0.147068
epoch=450 loss=0.130592

Predictions:
-0.0299
0.3383
0.6577

Weights:
wx = -0.38864016983809346
wh = 0.14093215455676597
wy = -0.9944485250072372
b  = 0.4186921962241345


In [ ]:
class RNN:

    def __init__(self,input_size,hidden_size,output_size):

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size

        self.Wxh = np.random.randn(hidden_size,input_size) * 0.1

        self.Whh = np.random.randn(hidden_size,hidden_size) * 0.1

        self.Why = np.random.randn(output_size,hidden_size) * 0.1

        self.bh = np.zeros((hidden_size, 1))

        self.by = np.zeros((output_size, 1))

    def forward(self, xs):

        self.xs = xs

        self.hs = {}
        self.zs = {}
        self.ys = {}

        self.hs[-1] = np.zeros((self.hidden_size, 1))

        for t in range(len(xs)):

            x = xs[t]

            z = (self.Wxh @ x + self.Whh @ self.hs[t - 1]+ self.bh)

            h = np.tanh(z)

            y = (self.Why @ h + self.by)

            self.zs[t] = z
            self.hs[t] = h
            self.ys[t] = y

        return self.ys

    def loss(self, targets):

        loss = 0
        for t in range(len(targets)):
            diff = (self.ys[t]- targets[t])
            loss += (0.5* np.sum(diff ** 2))

        return loss

    def backward(self, targets):

        self.dWxh = np.zeros_like(self.Wxh)
        self.dWhh = np.zeros_like(self.Whh)
        self.dWhy = np.zeros_like(self.Why)
        self.dbh = np.zeros_like(self.bh)
        self.dby = np.zeros_like(self.by)
        dh_next = np.zeros((self.hidden_size, 1))

        T = len(targets)

        for t in reversed(range(T)):

            y = self.ys[t]
            target = targets[t]
            h = self.hs[t]

            h_prev = self.hs[t - 1]

            # dL/dy
            dy = y - target

            # output layer gradients
            self.dWhy += (dy @ h.T)
            self.dby += dy

            # gradient wrt hidden state
            dh = (self.Why.T @ dy) + dh_next

            # tanh derivative
            dz = dh * (1 - h * h)

            # recurrent gradients
            self.dWxh += (dz @ self.xs[t].T)
            self.dWhh += (dz @ h_prev.T)
            self.dbh += dz

            # send gradient backward
            dh_next = (self.Whh.T @ dz)

    def step(self, lr):
        self.Wxh -= lr * self.dWxh
        self.Whh -= lr * self.dWhh
        self.Why -= lr * self.dWhy
        self.bh -= lr * self.dbh
        self.by -= lr * self.dby



In [192]:
rnn = RNN(input_size=1,hidden_size=4,output_size=1)

xs = [
    np.array([[1.0]]),
    np.array([[2.0]]),
    np.array([[3.0]])
]

targets = [
    np.array([[0.0]]),
    np.array([[0.0]]),
    np.array([[1.0]])
]

for epoch in range(500):

    rnn.forward(xs)
    loss = rnn.loss(targets)
    rnn.backward(targets)
    rnn.step(0.01)

    if epoch % 50 == 0:

        print(f"epoch={epoch:3d} loss={loss:.6f}")

print("\nPredictions")
preds = rnn.forward(xs)
for t in range(len(xs)):
    print(preds[t].flatten())

epoch=  0 loss=0.570420
epoch= 50 loss=0.262976
epoch=100 loss=0.230270
epoch=150 loss=0.200934
epoch=200 loss=0.173992
epoch=250 loss=0.150105
epoch=300 loss=0.129818
epoch=350 loss=0.113331
epoch=400 loss=0.100417
epoch=450 loss=0.090505

Predictions
[-0.09444484]
[0.32297901]
[0.77082396]
